# **RQ2**: Dynamic Community Detection

In [ ]:
from cdlib import algorithms, TemporalClustering, LifeCycle
from cdlib import viz

# 1. Otteniamo la lista dei tempi presenti
times = sorted(list(g.temporal_snapshots_ids()))

# 2. INIZIALIZZAZIONE CONTENITORE TEMPORALE (istanza della classe TemporalClustering)
tc = TemporalClustering() 

# 3. DISCOVERY PER SNAPSHOT (Leiden)
print("Rilevamento comunità per snapshot...")

for t in times:
    # 1. Estrai lo snapshot grezzo
    Gt_dyn = g.time_slice(t, t)
    
    # 2. CONTROLLO DI SICUREZZA:
    # Se per caso lo snapshot è vuoto o None, saltalo
    if Gt_dyn is None or len(list(Gt_dyn.nodes())) == 0:
        print(f"Skipping time {t}: grafo vuoto")
        continue

    # 3. RICOSTRUZIONE MANUALE (La soluzione all'errore)
    # Creiamo un grafo NetworkX vuoto e lo popoliamo esplicitamente.
    # Questo bypassa qualsiasi incompatibilità di formato dell'oggetto 'Gt_dyn'.
    Gt = nx.Graph()
    Gt.add_nodes_from(list(Gt_dyn.nodes()))
    Gt.add_edges_from(list(Gt_dyn.edges()))
    
    # 4. Esegui Leiden
    # Ora Gt è sicuramente un oggetto NetworkX puro
    try:
        coms = algorithms.leiden(Gt) 
        tc.add_clustering(coms, t)
    except Exception as e:
        print(f"Errore su Leiden al tempo {t}: {e}")

print(f"Rilevati clustering per {len(tc.clusterings)} istanti temporali.")


# 4. MATCHING TEMPORALE (Calcolo esplicito dei flussi)
# Per l'Alluvial Plot, devi sapere quanto una comunità al tempo t 
# si sovrappone a una comunità al tempo t+1.

def get_temporal_flows(temporal_clustering, min_overlap=1):
    """
    Calcola i link (source, target, weight) tra snapshot consecutivi.
    Weight = numero di nodi in comune.
    """
    matches = []
    
    # Itera attraverso le coppie di tempi (t, t+1)
    sorted_times = sorted(temporal_clustering.clusterings.keys())
    
    for i in range(len(sorted_times) - 1):
        t_current = sorted_times[i]
        t_next = sorted_times[i+1]
        
        coms_current = temporal_clustering.clusterings[t_current]
        coms_next = temporal_clustering.clusterings[t_next]
        
        # Confronta ogni comunità di t con ogni comunità di t+1
        for cid_curr, nodes_curr in enumerate(coms_current.communities):
            set_curr = set(nodes_curr)
            
            for cid_next, nodes_next in enumerate(coms_next.communities):
                set_next = set(nodes_next)
                
                # Calcola intersezione (overlap)
                intersection = len(set_curr & set_next)
                
                # Se c'è overlap, salva il "flusso"
                if intersection >= min_overlap:
                    matches.append({
                        'source_time': t_current,
                        'target_time': t_next,
                        'source_com_id': f"T{t_current}_C{cid_curr}", # ID univoco
                        'target_com_id': f"T{t_next}_C{cid_next}",     # ID univoco
                        'weight': intersection, # Dimensione del flusso (per l'Alluvial)
                        'jaccard': intersection / len(set_curr | set_next) # Opzionale
                    })
    return matches

# Ottieni i dati per il plot
matches = get_temporal_flows(tc)

# Visualizza i primi 3 match trovati
print("\nEsempio di Matches (Flussi) trovati:")
for m in matches[:3]:
    print(m)

In [ ]:
# Alluvial Plot
events = LifeCycle(tc)
events.compute_events("facets")
fig = viz.plot_flow(events)
fig.show()